# HW2 - Seeded QFT / inverse-QFT Hardware Recovery

This assignment extends HW1 into a real quantum algorithm workflow. You will build a personalized Quantum Fourier Transform (QFT) recovery circuit, run it on a simulator, and optionally run a small version on IBM Quantum hardware.

The key idea is:

1. Prepare a seeded computational basis state.
2. Apply QFT.
3. Apply inverse QFT.
4. Measure.

If the QFT and inverse QFT are implemented correctly, the final measurement should recover the original input bitstring. The simulator should be clean. Real hardware may be noisy.

## What you submit

- `answers.json`
- optional: `qft_recovery_circuit.qpy`
- optional hardware screenshot/histogram if your instructor requests it

## AI-resilience / learning focus

This assignment uses seeded inputs, non-symmetric bitstrings, measurement mapping, bit-ordering interpretation, transpiler statistics, and optional hardware execution. A generic text-only answer is unlikely to know your exact output without running the notebook.

In [ ]:
# Setup cell for Google Colab / Jupyter
%pip -q install qiskit qiskit-aer qiskit-ibm-runtime matplotlib jsonschema

In [ ]:
import json
import hashlib
import math
from pathlib import Path

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

try:
    import qiskit
    print('Qiskit version:', qiskit.__version__)
except Exception:
    pass

## 1. Enter your student ID

Use your real assigned ID if this is being used in a course. For testing the template, you may use `demo_student`. Do not change the ID after generating outputs, because your seed and answer depend on it.

In [ ]:
STUDENT_ID = "demo_student"  # TODO: replace with your assigned ID
ASSIGNMENT_ID = "HW2"
SHOTS = 2048
HARDWARE_SHOTS = 4096

## 2. Generate your personalized configuration

This code gives you your seeded input state and measurement map. It does **not** directly give you the expected displayed bitstring. You must compute and verify that yourself.

In [ ]:
def stable_seed(student_id: str, assignment_id: str = "HW2") -> int:
    key = f"{assignment_id}|{student_id}".encode("utf-8")
    return int(hashlib.sha256(key).hexdigest()[:12], 16)


def bits_from_seed(seed: int, n: int):
    bits = [(seed >> (3 * i + 1)) & 1 for i in range(n)]
    if sum(bits) in (0, n) or bits == list(reversed(bits)):
        patterns = {3: [1, 0, 0], 4: [1, 0, 1, 1], 5: [1, 0, 1, 1, 0]}
        bits = patterns.get(n, [1 if i in (0, 2, n-1) else 0 for i in range(n)])
    return bits


def generate_student_config(student_id: str, assignment_id: str = "HW2"):
    seed = stable_seed(student_id, assignment_id)
    n = 3 + (seed % 3)
    input_bits = bits_from_seed(seed, n)
    measurement_mode = "direct" if ((seed >> 7) & 1) == 0 else "reversed"
    if measurement_mode == "direct":
        measurement_map = [[i, i] for i in range(n)]
    else:
        measurement_map = [[i, n - 1 - i] for i in range(n)]
    hardware_num_qubits = min(n, 3)
    hardware_input_bits = input_bits[:hardware_num_qubits]
    if hardware_input_bits == list(reversed(hardware_input_bits)):
        hardware_input_bits = [1, 0, 0][:hardware_num_qubits]
    return {
        "assignment_id": assignment_id,
        "student_id": student_id,
        "seed": seed,
        "num_qubits": n,
        "input_bits_q0_to_qn": input_bits,
        "measurement_mode": measurement_mode,
        "measurement_map": measurement_map,
        "hardware_num_qubits": hardware_num_qubits,
        "hardware_input_bits_q0_to_qn": hardware_input_bits,
        "hardware_measurement_map": [[i, i] for i in range(hardware_num_qubits)],
        "shots": SHOTS,
        "hardware_shots": HARDWARE_SHOTS,
    }

config = generate_student_config(STUDENT_ID, ASSIGNMENT_ID)
print(json.dumps(config, indent=2))

## 3. Implement QFT and inverse QFT

Complete the TODOs below. You should implement QFT using Hadamard gates, controlled phase rotations, and final swaps. Then implement inverse QFT as the inverse operation.

Do not replace QFT + inverse QFT with the identity circuit. The purpose is to build and inspect the actual QFT workflow.

In [ ]:
def build_qft_circuit(num_qubits: int, inverse: bool = False):
    qc = QuantumCircuit(num_qubits, name="IQFT" if inverse else "QFT")

    if not inverse:
        # TODO: implement QFT.
        # Suggested structure:
        # for target in reversed(range(num_qubits)):
        #     apply H(target)
        #     apply controlled phase rotations from lower-index controls to target
        # then apply swaps
        for target in reversed(range(num_qubits)):
            qc.h(target)
            for control in reversed(range(target)):
                angle = math.pi / (2 ** (target - control))
                qc.cp(angle, control, target)
        for i in range(num_qubits // 2):
            qc.swap(i, num_qubits - 1 - i)
    else:
        # TODO: implement inverse QFT.
        # It should undo the QFT above.
        for i in range(num_qubits // 2):
            qc.swap(i, num_qubits - 1 - i)
        for target in range(num_qubits):
            for control in range(target):
                angle = -math.pi / (2 ** (target - control))
                qc.cp(angle, control, target)
            qc.h(target)

    return qc

# Quick visual check
build_qft_circuit(config["num_qubits"], inverse=False).draw("mpl")

## 4. Build your QFT recovery circuit

Your circuit must:

1. Prepare your seeded input bits in `q[0]...q[n-1]` order.
2. Apply QFT.
3. Apply inverse QFT.
4. Measure using `measurement_map`.

Because QFT followed by inverse QFT should recover the original input, the expected logical output is the same as your input bits, but you still need to account for measurement mapping and Qiskit's displayed count order.

In [ ]:
n = config["num_qubits"]
q = QuantumRegister(n, "q")
c = ClassicalRegister(n, "c")
qc = QuantumCircuit(q, c)

# Prepare seeded input state
for i, bit in enumerate(config["input_bits_q0_to_qn"]):
    if bit == 1:
        qc.x(q[i])

qc.barrier(label="prepare")
qc.compose(build_qft_circuit(n, inverse=False), qubits=list(range(n)), inplace=True)
qc.barrier(label="after_qft")
qc.compose(build_qft_circuit(n, inverse=True), qubits=list(range(n)), inplace=True)
qc.barrier(label="after_iqft")

# Measure according to your personalized map
for q_index, c_index in config["measurement_map"]:
    qc.measure(q[q_index], c[c_index])

qc.draw("mpl")

## 5. Compute the expected displayed bitstring

You must compute two strings:

- `expected_logical`: logical qubit string in `q[n-1]...q[0]` order.
- `expected_display`: Qiskit displayed count string in `c[n-1]...c[0]` order.

Hints:

1. The logical output after QFT + inverse QFT should equal your original input bits.
2. Use `measurement_map` to place each final qubit value into its measured classical bit.
3. Qiskit count keys are displayed as `c[n-1]...c[0]`.

In [ ]:
# TODO: compute these yourself.
# Replace the placeholder code if you want to practice. The logic is intentionally visible here
# because this is a student template; for a stricter version, the instructor could turn this into blanks.

final_bits_q0_to_qn = list(config["input_bits_q0_to_qn"])

classical_bits = [0] * n
for q_index, c_index in config["measurement_map"]:
    classical_bits[c_index] = final_bits_q0_to_qn[q_index]

expected_display = "".join(str(classical_bits[i]) for i in reversed(range(n)))
expected_logical = "".join(str(final_bits_q0_to_qn[i]) for i in reversed(range(n)))

print("Expected logical qubit string q[n-1]...q[0]:", expected_logical)
print("Expected displayed count string c[n-1]...c[0]:", expected_display)

## 6. Run the simulator and collect counts

The ideal simulator should put all shots on the expected displayed bitstring. If it does not, check your QFT/inverse QFT, input preparation, and measurement mapping.

In [ ]:
sim = AerSimulator(seed_simulator=config["seed"] % (2**32 - 1))
tqc = transpile(qc, sim, seed_transpiler=config["seed"] % (2**32 - 1), optimization_level=0)
result = sim.run(tqc, shots=SHOTS).result()
counts = {str(k): int(v) for k, v in result.get_counts().items()}
dominant_bitstring = max(counts, key=counts.get)

print("Counts:", counts)
print("Dominant bitstring:", dominant_bitstring)
print("Expected displayed bitstring:", expected_display)
print("Match?", dominant_bitstring == expected_display)
plot_histogram(counts, title="HW2 simulator: QFT followed by inverse QFT")

## 7. Record circuit/transpiler statistics

These statistics help show that you actually built and transpiled a circuit. Later assignments and hardware runs will use these as part of the AI-resilience layer.

In [ ]:
original_depth = qc.depth()
transpiled_depth_simulator = tqc.depth()
operation_counts_simulator = {str(k): int(v) for k, v in tqc.count_ops().items()}

print("Original depth:", original_depth)
print("Transpiled simulator depth:", transpiled_depth_simulator)
print("Simulator operation counts:", operation_counts_simulator)

## 8. Optional IBM Quantum hardware run

This section is optional for autograding, but important for the research project. It runs a small 3-qubit version on real IBM hardware when access is available.

You need an IBM Quantum token. Do not hard-code your token into a notebook you plan to share publicly.

In [ ]:
# Optional hardware setup. Uncomment and run only if you have IBM Quantum access.

# %pip -q install qiskit-ibm-runtime
# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
#
# # Save your token once, or load an already saved account.
# # QiskitRuntimeService.save_account(channel='ibm_quantum_platform', token='YOUR_TOKEN', overwrite=True)
# service = QiskitRuntimeService(channel='ibm_quantum_platform')
# backend = service.least_busy(operational=True, simulator=False, min_num_qubits=config['hardware_num_qubits'])
# print('Selected backend:', backend.name)

In [ ]:
# Optional: build and run the small hardware circuit.

# hw_config = dict(config)
# hw_config['num_qubits'] = config['hardware_num_qubits']
# hw_config['input_bits_q0_to_qn'] = config['hardware_input_bits_q0_to_qn']
# hw_config['measurement_map'] = config['hardware_measurement_map']
#
# hn = hw_config['num_qubits']
# hq = QuantumRegister(hn, 'q')
# hc = ClassicalRegister(hn, 'c')
# hw_qc = QuantumCircuit(hq, hc)
# for i, bit in enumerate(hw_config['input_bits_q0_to_qn']):
#     if bit == 1:
#         hw_qc.x(hq[i])
# hw_qc.barrier(label='prepare')
# hw_qc.compose(build_qft_circuit(hn, inverse=False), qubits=list(range(hn)), inplace=True)
# hw_qc.barrier(label='after_qft')
# hw_qc.compose(build_qft_circuit(hn, inverse=True), qubits=list(range(hn)), inplace=True)
# hw_qc.barrier(label='after_iqft')
# for q_index, c_index in hw_config['measurement_map']:
#     hw_qc.measure(hq[q_index], hc[c_index])
#
# hardware_original_depth = hw_qc.depth()
# pm = generate_preset_pass_manager(backend=backend, optimization_level=1, seed_transpiler=config['seed'] % (2**32 - 1))
# isa_circuit = pm.run(hw_qc)
# hardware_transpiled_depth = isa_circuit.depth()
# hardware_operation_counts = {str(k): int(v) for k, v in isa_circuit.count_ops().items()}
# print('Hardware original depth:', hardware_original_depth)
# print('Hardware transpiled depth:', hardware_transpiled_depth)
# print('Hardware operation counts:', hardware_operation_counts)
#
# sampler = Sampler(mode=backend)
# job = sampler.run([isa_circuit], shots=HARDWARE_SHOTS)
# print('Job ID:', job.job_id())
# print('Status:', job.status())

In [ ]:
# Optional: retrieve hardware results after the job completes.

# result = job.result()
# pub_result = result[0]
# try:
#     hardware_counts = pub_result.data.c.get_counts()
# except Exception:
#     hardware_counts = None
#     for field_name in dir(pub_result.data):
#         if field_name.startswith('_'):
#             continue
#         field = getattr(pub_result.data, field_name)
#         if hasattr(field, 'get_counts'):
#             hardware_counts = field.get_counts()
#             print(f'Counts extracted from classical register: {field_name}')
#             break
#     if hardware_counts is None:
#         raise RuntimeError('Could not extract counts. Inspect pub_result.data to find the classical register name.')
#
# hardware_counts = {str(k): int(v) for k, v in hardware_counts.items()}
# hardware_dominant_bitstring = max(hardware_counts, key=hardware_counts.get)
# hardware_expected_display = None
# hw_bits = hw_config['input_bits_q0_to_qn']
# hw_classical_bits = [0] * hn
# for q_index, c_index in hw_config['measurement_map']:
#     hw_classical_bits[c_index] = hw_bits[q_index]
# hardware_expected_display = ''.join(str(hw_classical_bits[i]) for i in reversed(range(hn)))
# hardware_expected_percentage = hardware_counts.get(hardware_expected_display, 0) / HARDWARE_SHOTS * 100
#
# print('Hardware counts:', hardware_counts)
# print('Expected hardware display:', hardware_expected_display)
# print('Dominant hardware bitstring:', hardware_dominant_bitstring)
# print('Expected bitstring percentage:', hardware_expected_percentage)
# plot_histogram(hardware_counts, title=f'IBM hardware: HW2 QFT recovery on {backend.name}')

## 9. Reflections

Answer briefly but specifically. These reflections are part of the learning and AI-resilience design.

In [ ]:
reflection_qft_recovery = """QFT followed by inverse QFT should recover the original computational basis state. In the simulator, the dominant measured bitstring should match the expected displayed bitstring exactly because the ideal simulator has no hardware noise."""

reflection_bit_order = """Qiskit displays count strings in classical-bit order c[n-1]...c[0]. I used the measurement map to determine which qubit value was stored in each classical bit before forming the displayed bitstring."""

reflection_hardware_noise = """On real IBM hardware, the expected bitstring may still be dominant, but other bitstrings can appear because of gate errors, readout errors, routing/transpilation overhead, and general device noise. If the expected bitstring is not clearly dominant, the hardware result may be partial or inconclusive."""

## 10. Export answers.json

Run this cell after completing the simulator section. If you also ran hardware, fill in the hardware variables before export. If you did not run hardware, the hardware fields may remain `None`.

In [ ]:
# Default hardware fields if you did not run the optional hardware section.
hardware_attempted = 'hardware_counts' in globals()
hardware_backend = backend.name if 'backend' in globals() else None
hardware_job_id = job.job_id() if 'job' in globals() else None
hardware_counts_out = hardware_counts if 'hardware_counts' in globals() else None
hardware_dominant_out = hardware_dominant_bitstring if 'hardware_dominant_bitstring' in globals() else None
hardware_expected_percentage_out = hardware_expected_percentage if 'hardware_expected_percentage' in globals() else None
hardware_original_depth_out = hardware_original_depth if 'hardware_original_depth' in globals() else None
hardware_transpiled_depth_out = hardware_transpiled_depth if 'hardware_transpiled_depth' in globals() else None
hardware_operation_counts_out = hardware_operation_counts if 'hardware_operation_counts' in globals() else None

answers = {
    "assignment_id": ASSIGNMENT_ID,
    "student_id": STUDENT_ID,
    "seed": config["seed"],
    "num_qubits": config["num_qubits"],
    "input_bits_q0_to_qn": config["input_bits_q0_to_qn"],
    "measurement_mode": config["measurement_mode"],
    "measurement_map": config["measurement_map"],
    "expected_logical_qn_to_q0": expected_logical,
    "expected_display_bitstring": expected_display,
    "shots": SHOTS,
    "counts": counts,
    "dominant_bitstring": dominant_bitstring,
    "original_depth": original_depth,
    "transpiled_depth_simulator": transpiled_depth_simulator,
    "operation_counts_simulator": operation_counts_simulator,
    "hardware_attempted": hardware_attempted,
    "hardware_backend": hardware_backend,
    "hardware_job_id": hardware_job_id,
    "hardware_counts": hardware_counts_out,
    "hardware_dominant_bitstring": hardware_dominant_out,
    "hardware_expected_percentage": hardware_expected_percentage_out,
    "hardware_original_depth": hardware_original_depth_out,
    "hardware_transpiled_depth": hardware_transpiled_depth_out,
    "hardware_operation_counts": hardware_operation_counts_out,
    "reflection_qft_recovery": reflection_qft_recovery,
    "reflection_bit_order": reflection_bit_order,
    "reflection_hardware_noise": reflection_hardware_noise,
}

with open("answers.json", "w") as f:
    json.dump(answers, f, indent=2)

print(json.dumps(answers, indent=2))

## 11. Optional QPY export

If requested, export your circuit object for instructor-side validation.

In [ ]:
# Optional QPY export
from qiskit import qpy

with open("qft_recovery_circuit.qpy", "wb") as f:
    qpy.dump(qc, f)

print("Saved qft_recovery_circuit.qpy")